<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml_v2/cours/seance2_cours.ipynb)

# Séance 4.2 — Prédire une décision — qui va résilier ?

**Cours** · durée : 4h (2h de cours, 2h d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- préparer des données pour un problème de **classification**
- construire un modèle qui estime une **probabilité de résiliation**
- transformer cette probabilité en décision à l'aide d'un **seuil**
- comprendre les différentes erreurs possibles grâce à la **matrice de confusion**
- choisir une mesure d'évaluation adaptée parmi la justesse, la précision, le rappel et le F1
- comprendre pourquoi le choix du seuil dépend du **contexte métier** et du coût des erreurs

Dans cette séance, nous cherchons à répondre à une question simple :

> **À partir des informations disponibles sur un abonné, peut-on estimer son risque de résiliation et utiliser cette estimation pour prendre une décision ?**

L'objectif n'est pas seulement de savoir entraîner un modèle, mais de comprendre comment passer :

**des données → à une probabilité → à une décision.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/bloc4_ml/data/"

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")
tel.head()

## 1. Le problème

Une ligne correspond à un abonné.

La variable `churn` indique ce qui s'est passé :

- `0` : l'abonné est resté ;
- `1` : l'abonné a résilié.

La question prédictive est :

> **À partir des informations disponibles sur un abonné, quel est son risque de résiliation ?**

C'est un problème d'**apprentissage supervisé** et, puisque la cible est une catégorie 0/1, un problème de **classification**.

In [ ]:
print("Nombre de lignes :", len(tel))
print("Taux brut de résiliation :", round(100 * tel["churn"].mean(), 1), "%")

Le taux de résiliation donne déjà une information importante : les deux classes ne sont pas nécessairement présentes dans les mêmes proportions.

Cela aura des conséquences sur l'évaluation du modèle.

## 2. Préparer les données

## Une colonne numérique stockée comme texte

La colonne `total` contient quelques valeurs qui ne peuvent pas être converties directement en nombre.

On les transforme en valeurs manquantes pour pouvoir les inspecter.

In [ ]:
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")

print("Valeurs manquantes dans total :", tel["total"].isna().sum())

Dans ce fichier, ces lignes correspondent à quelques abonnés très récents sans facture cumulée exploitable.

Pour la suite, nous retirons ces lignes.

In [ ]:
tel = tel.dropna(subset=["total"]).copy()

## Une première lecture descriptive

Avant d'ajuster un modèle, on regarde si certains profils présentent des taux de résiliation différents.

Par exemple, selon le type de contrat :

In [ ]:
if "contrat" in tel.columns:
    taux_contrat = (
        tel.groupby("contrat")["churn"]
           .mean()
           .mul(100)
           .round(1)
    )
    print(taux_contrat)

Une différence de taux peut être utile pour la **prédiction**.

Elle ne prouve pas qu'un type de contrat **cause** la résiliation.

## 3. Transformer les variables pour le modèle


Nous voulons prédire la variable `churn`.

On sépare donc :

- **`y`** : la variable à prédire, ici `churn` ;
- **`X`** : les variables que le modèle utilisera pour faire cette prédiction.

In [ ]:
y = tel["churn"]

id_cols = [c for c in ["client_id", "customerID"] if c in tel.columns]

X_brut = tel.drop(columns=["churn"] + id_cols)

X = pd.get_dummies(
    X_brut,
    drop_first=True,
    dtype=int
)

print("Dimensions de X :", X.shape)
X.head()

## Que fait `get_dummies` ?

Supposons :

```text
contrat = mensuel / un_an / deux_ans
```

Après encodage, on peut obtenir :

```text
contrat_un_an
contrat_deux_ans
```

Chaque colonne vaut 0 ou 1.

Quand les deux valent 0, on retrouve la modalité de référence : ici `mensuel`.

`drop_first=True` évite de créer des colonnes parfaitement redondantes.

`dtype=int` permet d'obtenir des 0 et des 1 au lieu de booléennes ("True" ou "False").

## 4. Train / test et `stratify`

Comme en 4.1, le jeu de test doit rester à l'écart de l'apprentissage.

Ici, on ajoute :

```python
stratify=y
```

pour conserver approximativement la même proportion de churn dans le train et le test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Taux de churn train :", round(100 * y_train.mean(), 1), "%")
print("Taux de churn test  :", round(100 * y_test.mean(), 1), "%")

Sans stratification, un tirage aléatoire peut produire des proportions un peu différentes.

Ce n'est pas toujours grave, mais sur une cible minoritaire, conserver des proportions proches rend l'évaluation plus stable et plus lisible.

## 5. Pourquoi une régression logistique ?

Une régression linéaire peut produire n'importe quel nombre, par exemple `-0.2` ou `1.3`.

Pour un risque, nous voulons une valeur comprise entre 0 et 1.

La régression logistique transforme au
contraire un score linéaire en une probabilité valide grâce à la fonction
logistique :

$$
P(y=1\mid X)=\frac{1}{1+e^{-(\beta_0+\beta_1X_1+\cdots+\beta_pX_p)}}.
$$

Un coefficient logistique ne se lit donc pas comme « `y` augmente directement
de tant d'unités ».
Enfin, le modèle ne décide pas seul : le passage de la probabilité à
« résilie » ou « reste » dépend d'un **seuil**, dont le choix doit refléter le
coût des différentes erreurs.

## Pourquoi `StandardScaler` et `make_pipeline` ?

Les variables numériques peuvent avoir des échelles très différentes :

- ancienneté : quelques dizaines de mois ;
- montant total : plusieurs milliers d'euros.

`StandardScaler()` remet les variables numériques sur des échelles comparables pour faciliter l'ajustement de la régression logistique.

`make_pipeline(...)` enchaîne automatiquement :

1. la standardisation ;
2. le modèle.

Le même enchaînement sera appliqué au train et au test.

In [ ]:
modele = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

modele.fit(X_train, y_train)

## 6. Le modèle produit d'abord une probabilité

`predict_proba()` renvoie une probabilité pour chaque classe.

La colonne `[:, 1]` correspond à la probabilité de la classe `1`, donc ici au risque de résiliation.

In [ ]:
proba = modele.predict_proba(X_test)[:, 1]

pd.Series(proba).describe().round(3)

Exemple :

- `0.12` : risque estimé de 12 % ;
- `0.53` : risque estimé de 53 % ;
- `0.81` : risque estimé de 81 %.

Une probabilité de 53 % ne signifie pas que l'abonné va « certainement » partir.

Elle exprime le niveau de risque estimé par le modèle pour ce profil.

In [ ]:
exemple = pd.DataFrame({
    "probabilite_predite": proba[:10],
    "churn_reel": y_test.iloc[:10].to_numpy()
})

exemple.round(3)

## 7. Le seuil transforme le risque en classe

Il faut maintenant décider à partir de quel niveau de risque on prédit `1`.

Avec un seuil de 0,50 :

```python
pred = (proba > 0.50).astype(int)
```

- au-dessus de 0,50 → `1` ;
- sinon → `0`.

In [ ]:
seuil = 0.50
pred = (proba > seuil).astype(int)

pd.DataFrame({
    "proba": proba[:8],
    "classe_predite": pred[:8]
}).round(3)

Le seuil n'est **pas appris automatiquement** comme une vérité absolue.

Il traduit une règle de décision.

Deux abonnés à 0,49 et 0,51 ont des risques très proches, même s'ils tombent de part et d'autre du seuil de 0,50.

## 8. La matrice de confusion

Une classification binaire peut se tromper de deux manières.

|  | prédit : reste | prédit : part |
|---|---|---|
| **reste vraiment** | **vrai négatif (VN)** |  **faux positif (FP)** |
| **part vraiment** |  **faux négatif (FN)** |  **vrai positif (VP)** |

Dans notre contexte :

- **FP** : on cible un abonné qui serait resté ;
- **FN** : on laisse passer un abonné qui résilie réellement.

In [ ]:
cm = confusion_matrix(y_test, pred)

pd.DataFrame(
    cm,
    index=["réel : reste", "réel : part"],
    columns=["prédit : reste", "prédit : part"]
)

La matrice de confusion est souvent plus informative qu'un score unique, car elle montre **quel type d'erreur** est commis.

## 9. Calculer justesse, précision, rappel et F1

| Mesure | On divise | Elle répond à |
|---|---|---|
| **justesse** | (VN + VP) / total | quelle part de mes prédictions est bonne ? |
| **précision** | VP / (VP + FP) | parmi ceux que j'**appelle**, combien partaient vraiment ? |
| **rappel** | VP / (VP + FN) | parmi ceux qui **partent**, combien ai-je récupéré ? |
| **F1** | 2 × précision × rappel / (précision + rappel) | les deux tiennent-elles ensemble ? |

La précision se divise par ceux qu'on **appelle**, le rappel par ceux qui
**partent**. C'est tout ce qui les sépare, et c'est ce qui change tout.

In [ ]:
scores = pd.Series({
    "justesse": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "rappel": recall_score(y_test, pred, zero_division=0),
    "F1": f1_score(y_test, pred, zero_division=0),
})

scores.round(3)

## Comment les interpréter ?

### Justesse
> Parmi **toutes** les prédictions, quelle proportion est correcte ?

### Précision
> Parmi les abonnés que le modèle classe « à risque », quelle proportion résilie réellement ?

### Rappel
> Parmi **tous les abonnés qui résilient réellement**, quelle proportion le modèle a-t-il repérée ?

### F1
Combine précision et rappel dans un seul score.  
Il est utile pour résumer leur compromis, mais ne connaît pas le coût économique réel des erreurs.

Repère simple :

- **précision** → partir de ceux que le modèle a déclarés positifs ;
- **rappel** → partir des positifs réels.

## 10. Le piège de la justesse

Si environ trois quarts des abonnés restent, un modèle qui prédit toujours « reste » obtient déjà une justesse élevée.

Vérifions :

In [ ]:
pred_majoritaire = np.zeros(len(y_test), dtype=int)

print("Justesse :", round(accuracy_score(y_test, pred_majoritaire), 3))
print("Rappel   :", round(recall_score(y_test, pred_majoritaire, zero_division=0), 3))
print("F1       :", round(f1_score(y_test, pred_majoritaire, zero_division=0), 3))

Le modèle « toujours reste » peut donc avoir une justesse respectable tout en étant **incapable de retrouver un seul partant**.

Sur une cible déséquilibrée, la justesse ne doit jamais être regardée seule.

## 11. Changer le seuil change le compromis

Le modèle reste exactement le même.

Nous changeons seulement la règle qui transforme une probabilité en classe.

In [ ]:
lignes = []

for seuil in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    p = (proba > seuil).astype(int)

    lignes.append({
        "seuil": seuil,
        "contacts": int(p.sum()),
        "precision": precision_score(y_test, p, zero_division=0),
        "rappel": recall_score(y_test, p, zero_division=0),
        "F1": f1_score(y_test, p, zero_division=0),
    })

comparaison = pd.DataFrame(lignes)
comparaison.round(3)

En général :

- **baisser le seuil** → plus de personnes classées positives → rappel plus élevé, mais davantage de faux positifs ;
- **monter le seuil** → moins de personnes classées positives → plus de vrais partants risquent d'être manqués.

Le « bon » seuil dépend donc de ce que l'on cherche à éviter.

## 12. Une décision métier : combien coûte chaque erreur ?

Supposons :

- un contact coûte **15 €** ;
- un client retenu représente **300 €** de marge annuelle ;
- parmi les vrais partants contactés, **30 %** acceptent l'offre et restent.

Ce sont des **hypothèses de gestion**, pas des résultats du modèle.

In [ ]:
def gain_estime(seuil):
    p = (proba > seuil).astype(int)

    contacts = int(p.sum())

    vrais_partants_contactes = int(
        ((p == 1) & (y_test.to_numpy() == 1)).sum()
    )

    gain = (
        vrais_partants_contactes * 0.30 * 300
        - contacts * 15
    )

    return {
        "seuil": seuil,
        "contacts": contacts,
        "vrais_partants_contactes": vrais_partants_contactes,
        "gain_estime": gain,
    }

resultats = pd.DataFrame(
    [gain_estime(s) for s in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]]
)

resultats.round({
    "seuil": 2,
    "gain_estime": 0
})

Le seuil économiquement intéressant n'est pas nécessairement celui qui maximise la justesse ou le F1.

Il dépend :

- du coût d'un contact inutile ;
- du coût d'un client perdu ;
- de la capacité de l'équipe à contacter les clients ;
- de l'efficacité réelle de l'action de rétention.

## 13. Prédire n'est pas mesurer l'effet d'une action

Le modèle de churn répond à :

> **« Qui risque de partir ? »**

Il ne répond pas directement à :

> **« Qui resterait grâce à notre appel ? »**

Un client à haut risque peut être impossible à convaincre.  
Un client à risque modéré peut au contraire très bien répondre à l'offre.

Mesurer l'effet propre d'un appel est une **question causale** : ce sera l'objet du bloc 5.

## 14. Dernier contrôle : éviter la fuite de données

Comme en 4.1, une variable n'est utilisable que si elle est disponible **avant** la décision.

Exemples de variables problématiques :

- une information enregistrée après la résiliation ;
- le motif final de clôture ;
- une variable calculée à partir d'événements futurs ;
- la cible `churn` elle-même.

Un score élevé obtenu avec une information future n'est pas une bonne prédiction : c'est une fuite de données.

## 15. Synthèse

Le workflow de classification est :

```text
préparer X et y
      ↓
encoder les variables qualitatives
      ↓
séparer train / test
      ↓
fit sur le train
      ↓
predict_proba sur le test
      ↓
choisir un seuil
      ↓
matrice de confusion + métriques
      ↓
relier les erreurs à la décision métier
```

Aide-mémoire des commandes

| Vous voulez... | La commande |
|---|---|
| du texte en colonnes numériques | `pd.get_dummies(X, drop_first=True)` |
| enchaîner mise à l'échelle et modèle | `make_pipeline(StandardScaler(), LogisticRegression())` |
| une décision (0 ou 1) | `m.predict(X_te)` |
| une **probabilité** | `m.predict_proba(X_te)[:, 1]` |
| une décision à partir d'une probabilité | `(proba > 0.20).astype(int)` |
| compter ceux qu'on appelle **et** qui partaient | `((p == 1) & (y_te == 1)).sum()` |
| écrire un calcul une fois et le rejouer | `def gain(seuil): ... return ...` |
| la matrice de confusion | `confusion_matrix(y_te, pred)` |
| la part de vrais parmi les prédits partants | `precision_score(y_te, pred)` |
| la part de partants retrouvés | `recall_score(y_te, pred)` |
| les deux à la fois, en un seul nombre | `f1_score(y_te, pred)` |

### La matrice de confusion, en clair

|  | prédit : reste | prédit : part |
|---|---|---|
| **reste vraiment** | vrai négatif — bien vu | **faux positif** : un appel pour rien |
| **part vraiment** | **faux négatif** : client perdu sans rien tenter | vrai positif — bien vu |

### Les quatre mesures

| Mesure | Ce qu'elle divise | Ce qu'elle répond |
|---|---|---|
| justesse | (VN + VP) / total | quelle part de mes prédictions sont bonnes ? |
| précision | VP / (VP + FP) | parmi ceux que j'appelle, combien partaient vraiment ? |
| rappel | VP / (VP + FN) | parmi ceux qui partent, combien ai-je retrouvés ? |
| F1 | 2 × précision × rappel / (précision + rappel) | les deux se tiennent-elles ensemble ? |

### Les trois phrases à retenir

1. **La justesse est un piège.** Prédire « personne ne part » donne 73,4 % de
   bonnes réponses, zéro client sauvé — et un F1 de 0.

2. **Précision et rappel arbitrent deux coûts différents.** Le rappel dit
   combien de partants on retrouve, la précision combien de contacts sont
   utiles. On ne maximise pas les deux ; le F1 dit où on en est des deux.

3. **Le seuil est une décision de gestion.** Le déplacer de 0,50 à 0,20 fait
   passer le gain de la campagne de 20 370 € à 28 365 € — sans changer une
   ligne du modèle.